In [16]:
"""
Foundings:

type mismatches:
    salary -> int
    created_at -> datetime
    deadline_at -> datetime
    view_count -> int after parsing 
    
6 nulls in deadline_at column, these are outdated listings that out of the main scrape range. Drop during cleaning.
"Company" placeholder in company name column. Valid listings, but replace with null to not bias company-based analysis
almost 70 percent of salary column is 0, convert to null
every value in direct_apply is 0, drop the column
company_id, url, fetched_at is irrelevant to the purpose of this analysis, drop these columns.
description needs extraction
"""

'\nFoundings:\n\ntype mismatches:\n    salary -> int\n    created_at -> datetime\n    deadline_at -> datetime\n    view_count -> int after parsing \n\n6 nulls in deadline_at column, these are outdated listings that out of the main scrape range. Drop during cleaning.\n"Company" placeholder in company name column. Valid listings, but replace with null to not bias company-based analysis\nalmost 70 percent of salary column is 0, convert to null\nevery value in direct_apply is 0, drop the column\ncompany_id, url, fetched_at is irrelevant to the purpose of this analysis, drop these columns.\ndescription needs extraction\n'

In [17]:
import pandas as pd

from jobanalysis.paths import RAW_DATA
from jobanalysis.io import load

In [18]:
df = load(RAW_DATA)

df.head()

,id,title,company,company_id,salary,category,created_at,deadline_at,view_count,direct_apply,url,description,fetched_at
0,85507,Maliyyə şöbəsinin müdiri,AzerMaya MMC,14385,0,Maliyyə xidmətləri,2024-09-03T00:00:00+04:00,NaN,3.3K,0,https://jobsearch.az/vacancies/azer-maya-isteh...,Tələblər: \n\n \n\t Ali təhsil \n\t Maliy...,2026-05-04 18:11:57
1,85668,General Assistant,Turan Drilling and Engineering Company,7173,0,"Inzibati, Biznes və İdarəetmə",2024-09-04T00:00:00+04:00,NaN,6.6K,0,https://jobsearch.az/vacancies/turan-drilling-...,A vacancy has been raised externally for a Gen...,2026-05-04 18:11:57
2,86834,Kabel satışı üzrə mütəxəssis,Baku Cable Goknur,2722,0,Pərakəndə satış və müştəri xidmətləri,2024-09-16T00:00:00+04:00,NaN,1.2K,0,https://jobsearch.az/vacancies/baku-cable-gokn...,Bakı cable Göknur şirkətində K abel satışı üzr...,2026-05-04 18:11:57
3,86880,Sales and Purchasing Specialist,Bureau Veritas Azerbaijan,3068,0,Pərakəndə satış və müştəri xidmətləri,2024-09-17T00:00:00+04:00,NaN,2.6K,0,https://jobsearch.az/vacancies/bureau-veritas-...,"Created in 1828, Bureau Veritas is a global le...",2026-05-04 18:11:57
4,87172,Camaşırxana xidmətçisi,Baku Marriott Hotel Boulevard,2761,0,Təsərrüfat və əmlak xidmətləri,2024-09-19T00:00:00+04:00,NaN,1.7K,0,https://jobsearch.az/vacancies/baku-marriott-h...,Şirkət : Baku Boulevard Hotel Company LLC \n Ü...,2026-05-04 18:11:57


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6872 entries, 0 to 6871
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            6872 non-null   int64 
 1   title         6872 non-null   str   
 2   company       6872 non-null   str   
 3   company_id    6872 non-null   int64 
 4   salary        6872 non-null   str   
 5   category      6872 non-null   str   
 6   created_at    6872 non-null   str   
 7   deadline_at   6866 non-null   str   
 8   view_count    6872 non-null   object
 9   direct_apply  6872 non-null   int64 
 10  url           6872 non-null   str   
 11  description   6872 non-null   str   
 12  fetched_at    6872 non-null   str   
dtypes: int64(3), object(1), str(9)
memory usage: 698.1+ KB


In [20]:
df.isna().sum()

id              0
title           0
company         0
company_id      0
salary          0
category        0
created_at      0
deadline_at     6
view_count      0
direct_apply    0
url             0
description     0
fetched_at      0
dtype: int64

In [21]:
df.describe()

,id,company_id,direct_apply
count,6872.000000,6872.000000,6872.0
mean,140348.071449,41886.741123,0.0
std,2893.248330,49721.241185,0.0
min,85507.000000,16.000000,0.0
25%,138240.750000,4002.000000,0.0
50%,140098.500000,7311.000000,0.0
75%,142724.250000,81758.000000,0.0
max,144545.000000,144516.000000,0.0


In [22]:
df["company"].value_counts()

company
Company                                        240
Azərmemarlayihə Dövlət Baş Layihə İnstitutu    157
Kontakt Home                                   138
IRES                                            96
PASHA Bank                                      65
                                              ... 
City Finance BOKT                                1
Mixpromax MMC                                    1
Клиника Система                                  1
Marvel Azerbaijan MMC                            1
Baku International School                        1
Name: count, Length: 1562, dtype: int64

In [23]:
df[df["company"] == "Company"].head()

,id,title,company,company_id,salary,category,created_at,deadline_at,view_count,direct_apply,url,description,fetched_at
7,136337,Office Admin/Office Manager,Company,103,0,"Inzibati, Biznes və İdarəetmə",2026-04-05T00:00:00+04:00,2026-05-05T00:00:00+04:00,1K,0,https://jobsearch.az/vacancies/company-office-...,Location: Cresent Offices \n\n Salary: Compe...,2026-05-04 18:11:57
16,136347,Trade Marketing Leader (FMCG),Company,103,0,"Marketinq, reklam, çap və nəşriyyat",2026-04-06T00:00:00+04:00,2026-05-06T00:00:00+04:00,344,0,https://jobsearch.az/vacancies/company-trade-m...,We are currently partnering with a leading FMC...,2026-05-04 18:11:57
112,136452,Gözətçi (Mühafizəçi),Company,103,900,"Təhlükəsizlik, uniforma və qoruyucu xidmətlər",2026-04-06T00:00:00+04:00,2026-05-06T00:00:00+04:00,799,0,https://jobsearch.az/vacancies/company-gozetci...,Qeyd - Fərdi yaşayış evi üçün gözətçi axtarıl...,2026-05-04 18:11:57
114,136454,Təlim və İnkişaf üzrə mütəxəssis və aparıcı mü...,Company,103,0,Təlim və tədris,2026-04-06T00:00:00+04:00,2026-05-06T00:00:00+04:00,546,0,https://jobsearch.az/vacancies/company-telim-v...,Vəzifə öhdəlikləri: \n\n \n\t Təlimlərin müəyy...,2026-05-04 18:11:57
118,136461,İnternet Klub üzrə Administrator,Company,103,0,"Otel, İaşə, Turizm",2026-04-06T00:00:00+04:00,2026-05-06T00:00:00+04:00,268,0,https://jobsearch.az/vacancies/company-interne...,Təcili Müasir kompyuter klubuna xanım işçi təl...,2026-05-04 18:11:57


In [24]:
df["salary"] = df["salary"].astype(int)
(df["salary"] == 0).sum() / len(df)

np.float64(0.7040162980209546)

In [25]:
(df["salary"].min(), df["salary"].max())

(np.int64(-3), np.int64(2147483647))

In [26]:
df["salary"].sort_values()

2354            -3
21               0
34               0
15               0
6850             0
           ...    
1632          4000
1039          4872
6762          5000
3080          6000
5425    2147483647
Name: salary, Length: 6872, dtype: int64

In [27]:
df["direct_apply"].value_counts()

direct_apply
0    6872
Name: count, dtype: int64

In [28]:
df["view_count"].apply(type).value_counts()

view_count
<class 'int'>    6614
<class 'str'>     258
Name: count, dtype: int64

In [29]:
df["category"].value_counts()

category
Inzibati, Biznes və İdarəetmə               818
Maliyyə xidmətləri                          687
Mühəndislik                                 563
Maliyyə, Bankçılıq və Sığorta               473
Pərakəndə satış və müştəri xidmətləri       428
                                           ... 
Əməliyyatların İdarə Edilməsi                 1
Ətraf Mühit və Əməyin Mühafizəsi              1
Elektrik, Elektronika və Avtomatlaşdırma      1
Mühəndislik Dəstəyi                           1
Mülki və Tikinti Mühəndisliyi                 1
Name: count, Length: 65, dtype: int64

In [30]:
pd.to_datetime(df["created_at"]).agg(["min", "max"])

min   2024-09-03 00:00:00+04:00
max   2026-07-01 00:00:00+04:00
Name: created_at, dtype: datetime64[us, UTC+04:00]